In [1]:
import numpy as np
import pandas as pd
import lightgbm as lgb

from numba import njit


def recall20(preds, targets, groups):
    total = 0
    nonempty = 0
    group_starts = np.cumsum(groups)

    for group_id in range(len(groups)):
        group_end = group_starts[group_id]
        group_start = group_end - groups[group_id]
        ranks = np.argsort(preds[group_start:group_end])[::-1]
        hits = 0
        for i in range(min(len(ranks), 20)):
            hits += targets[group_start + ranks[i]]

        actual = min(20, targets[group_start:group_end].sum())
        if actual > 0:
            total += hits / actual
            nonempty += 1

    return total / nonempty

# custom metric for LightGBM should return 
# "metric name", "metric value" and "greater is better" flag
def lgb_recall(preds, lgb_dataset):
    metric = recall20(preds, lgb_dataset.label, lgb_dataset.group)
    return 'recall@20', metric, True

### Numba version 

In [2]:
@njit() # the only difference from previous version
def numba_recall20(preds, targets, groups):
    total = 0
    nonempty = 0
    group_starts = np.cumsum(groups)

    for group_id in range(len(groups)):
        group_end = group_starts[group_id]
        group_start = group_end - groups[group_id]
        ranks = np.argsort(preds[group_start:group_end])[::-1]
        hits = 0
        for i in range(min(len(ranks), 20)):
            hits += targets[group_start + ranks[i]]

        actual = min(20, targets[group_start:group_end].sum())
        if actual > 0:
            total += hits / actual
            nonempty += 1

    return total / nonempty


def lgb_numba_recall(preds, lgb_dataset):
    metric = numba_recall20(preds, lgb_dataset.label, lgb_dataset.group)
    return 'numba_recall@20', metric, True



### Synthetic data

In [3]:
def create_dataset(size: int) -> lgb.Dataset:
    data = np.random.normal(size=(size, 10))
    target = np.random.randint(0, 2, size)
    groups = np.ones(size // 40, dtype=np.int32) * 40 # groups of 40
    return lgb.Dataset(data, target, group=groups)

In [4]:
size = 1_000_000

In [5]:
train_dataset = create_dataset(size)
eval_dataset = create_dataset(size)

### Built-in map@20 metric

In [6]:
%%time
params = {'objective': 'lambdarank', 'metric': 'map', 'eval_at': [20]}
model = lgb.train(
    params,
    train_dataset,
    valid_sets=eval_dataset,
    callbacks=[lgb.log_evaluation(5)],
)

[LightGBM] [Info] Total groups: 25000, total data: 1000000
[LightGBM] [Warning] Auto-choosing row-wise multi-threading, the overhead of testing was 0.030374 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2550
[LightGBM] [Info] Number of data points in the train set: 1000000, number of used features: 10
[LightGBM] [Info] Total groups: 25000, total data: 1000000
[5]	valid_0's map@20: 0.309931
[10]	valid_0's map@20: 0.309744
[15]	valid_0's map@20: 0.310568
[20]	valid_0's map@20: 0.310614
[25]	valid_0's map@20: 0.310793
[30]	valid_0's map@20: 0.31063
[35]	valid_0's map@20: 0.310951
[40]	valid_0's map@20: 0.311052
[45]	valid_0's map@20: 0.311436
[50]	valid_0's map@20: 0.311395
[55]	valid_0's map@20: 0.311441
[60]	valid_0's map@20: 0.311342
[65]	valid_0's map@20: 0.311036
[70]	valid_0's map@20: 0.311039
[75]	valid_0's map@20: 0.311096
[80]	valid_0's map@20: 0.31103
[85]	valid_0's 

### recall@20

In [7]:
%%time
params = {'objective': 'lambdarank', 'metric': '"None"'}
model = lgb.train(
    params,
    train_dataset,
    valid_sets=eval_dataset,
    feval=lgb_recall,
    callbacks=[lgb.log_evaluation(5)],
)

[LightGBM] [Warning] Auto-choosing col-wise multi-threading, the overhead of testing was 0.087870 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2550
[LightGBM] [Info] Number of data points in the train set: 1000000, number of used features: 10
[5]	valid_0's recall@20: 0.530979
[10]	valid_0's recall@20: 0.530791
[15]	valid_0's recall@20: 0.531601
[20]	valid_0's recall@20: 0.531058
[25]	valid_0's recall@20: 0.531554
[30]	valid_0's recall@20: 0.531634
[35]	valid_0's recall@20: 0.531831
[40]	valid_0's recall@20: 0.531879
[45]	valid_0's recall@20: 0.532626
[50]	valid_0's recall@20: 0.532562
[55]	valid_0's recall@20: 0.53237
[60]	valid_0's recall@20: 0.532311
[65]	valid_0's recall@20: 0.532034
[70]	valid_0's recall@20: 0.53182
[75]	valid_0's recall@20: 0.532
[80]	valid_0's recall@20: 0.531886
[85]	valid_0's recall@20: 0.531904
[90]	valid_0's recall@20: 0.532012
[95]	valid_0's recall@20: 0.532199
[100]	valid_0's recall@20: 0.532145
CPU times: 

### numba recall@20

In [8]:
%%time
# call function before benchmarking lightgbm, because we need numba to compile it
lgb_numba_recall(np.zeros(size), eval_dataset)

CPU times: user 2.62 s, sys: 48.9 ms, total: 2.66 s
Wall time: 2.77 s


('numba_recall@20', 0.5302279098967755, True)

In [9]:
%%time
params = {'objective': 'lambdarank', 'metric': '"None"'}
model = lgb.train(
    params,
    train_dataset,
    valid_sets=eval_dataset,
    feval=lgb_numba_recall,
    callbacks=[lgb.log_evaluation(5)],
)

[LightGBM] [Warning] Auto-choosing row-wise multi-threading, the overhead of testing was 0.028234 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2550
[LightGBM] [Info] Number of data points in the train set: 1000000, number of used features: 10
[5]	valid_0's numba_recall@20: 0.531244
[10]	valid_0's numba_recall@20: 0.531038
[15]	valid_0's numba_recall@20: 0.531525
[20]	valid_0's numba_recall@20: 0.531325
[25]	valid_0's numba_recall@20: 0.531482
[30]	valid_0's numba_recall@20: 0.53157
[35]	valid_0's numba_recall@20: 0.531777
[40]	valid_0's numba_recall@20: 0.531902
[45]	valid_0's numba_recall@20: 0.532458
[50]	valid_0's numba_recall@20: 0.532392
[55]	valid_0's numba_recall@20: 0.532458
[60]	valid_0's numba_recall@20: 0.532389
[65]	valid_0's numba_recall@20: 0.532022
[70]	valid_0's numba_recall@20: 0.531752
[75]	valid_0's numba_recall@20: 0.531928
[80]	valid_0's numba_recall@2

### Summary

In [10]:

from numba import prange

@njit(parallel=True) # added parallel flag
def numba_parallel_recall20(preds, targets, groups):
    total = 0
    nonempty = 0
    group_starts = np.cumsum(groups)

    for group_id in prange(len(groups)): # changed range to prange
        group_end = group_starts[group_id]
        group_start = group_end - groups[group_id]
        ranks = np.argsort(preds[group_start:group_end])[::-1]
        hits = 0
        for i in range(min(len(ranks), 20)):
            hits += targets[group_start + ranks[i]]

        actual = min(20, targets[group_start:group_end].sum())
        if actual > 0:
            total += hits / actual
            nonempty += 1

    return total / nonempty